In [1]:
import pandas as pd
import duckdb

df_device_status_daily = pd.DataFrame({
    "device_id": [
        "R05", "R05", "R05", "R05", "R05", "R05", "R05",
        "R16", "R16", "R16", "R16", "R16", "R16",
        "R34", "R34", "R34", "R34", "R34", "R34"
    ],
    "stat_date": [
        "2026-07-01",
        "2026-07-02",
        "2026-07-03",
        "2026-07-04",
        "2026-07-05",
        "2026-07-07",
        "2026-07-08",

        "2026-07-01",
        "2026-07-02",
        "2026-07-03",
        "2026-07-05",
        "2026-07-06",
        "2026-07-07",

        "2026-07-01",
        "2026-07-02",
        "2026-07-03",
        "2026-07-04",
        "2026-07-05",
        "2026-07-06"
    ],
    "status": [
        "OK",
        "ERROR",
        "ERROR",
        "ERROR",
        "OK",
        "ERROR",
        "ERROR",

        "ERROR",
        "ERROR",
        "OK",
        "ERROR",
        "ERROR",
        "ERROR",

        "ERROR",
        "ERROR",
        "ERROR",
        "ERROR",
        "OK",
        "ERROR"
    ]
})

df_device_status_daily["stat_date"] = pd.to_datetime(
    df_device_status_daily["stat_date"]
)

df_device_status_daily

,device_id,stat_date,status
0,R05,2026-07-01,OK
1,R05,2026-07-02,ERROR
2,R05,2026-07-03,ERROR
3,R05,2026-07-04,ERROR
4,R05,2026-07-05,OK
5,R05,2026-07-07,ERROR
6,R05,2026-07-08,ERROR
7,R16,2026-07-01,ERROR
8,R16,2026-07-02,ERROR
9,R16,2026-07-03,OK


# SQL Daily Review：连续 ERROR 天数区间

## 题目背景

设备每天生成一条状态记录。

状态包括：

- `OK`：设备正常；
- `ERROR`：设备异常。

现在需要识别每台设备连续处于 `ERROR` 状态的日期区间。

## 题目要求

找出每台设备中，**连续至少 3 个自然日处于 `ERROR` 状态的区间**。

### 连续判断规则

两条 `ERROR` 记录属于同一个异常区间，必须满足：

```text
当前日期 = 上一条 ERROR 日期 + 1 天
```

例如：

```text
2026-07-05  ERROR
2026-07-07  ERROR
```

虽然两条记录都是 `ERROR`，但中间缺少 `2026-07-06`，因此不能视为连续异常。

### 输出字段

| 字段 | 含义 |
|---|---|
| `device_id` | 设备编号 |
| `error_start_date` | 连续异常开始日期 |
| `error_end_date` | 连续异常结束日期 |
| `error_days` | 连续异常记录数 |

### 最终排序

按照以下顺序排列：

1. `device_id` 升序；
2. `error_start_date` 升序。

## 解题要求

- 先筛选 `status = 'ERROR'`；
- 使用 `LAG()` 获取上一条异常日期；
- 判断当前记录是否为新区间起点；
- 使用累计求和生成区间编号；
- 按照 `device_id` 和区间编号聚合；
- 只保留 `error_days >= 3` 的区间；
- 使用 CTE 分步骤完成。

In [13]:
query = """
WITH previous_table AS (
    SELECT
        device_id,
        stat_date,
        LAG(stat_date) OVER (
            PARTITION BY device_id
            ORDER BY stat_date
        ) AS previous_date
    FROM df_device_status_daily
    WHERE status = 'ERROR'
),

start_error_sign_table AS (
    SELECT
        device_id,
        stat_date,
        previous_date,
        CASE
            WHEN previous_date IS NULL
                 OR stat_date <> previous_date + INTERVAL 1 DAY
            THEN 1
            ELSE 0
        END AS error_start_sign
    FROM previous_table
),

phase_sign_table AS (
    SELECT
        device_id,
        stat_date,
        SUM(error_start_sign) OVER (
            PARTITION BY device_id
            ORDER BY stat_date
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        )::INTEGER AS phase_sign
    FROM start_error_sign_table
)

SELECT
    device_id,
    MIN(stat_date) AS error_start_date,
    MAX(stat_date) AS error_end_date,
    COUNT(*) AS error_days
FROM phase_sign_table
GROUP BY
    device_id,
    phase_sign
HAVING COUNT(*) >= 3
ORDER BY
    device_id,
    error_start_date;
"""

df = duckdb.execute(query).fetchdf()
df

,device_id,error_start_date,error_end_date,error_days
0,R05,2026-07-02,2026-07-04,3
1,R16,2026-07-05,2026-07-07,3
2,R34,2026-07-01,2026-07-04,4
